<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Intro_to_LLMs_Building_a_RAG_system_(Project_9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Retrieval-Augmented Generation (RAG) is a method that combines retrieval and generation:

**(1) Retrieval**

•	This is fetching relevant information from your own documents.

•	Done using vector embeddings + vector database.

•	Helps the model focus on relevant information instead of generating purely from training data.

**(2) Generation**


•	Uses a large language model (LLM) like GPT to generate answers based on retrieved content.

•	Ensures the answers are grounded in actual data, reducing hallucinations.

**Why it matters:**

•	A regular LLM can make up information.

•	RAG ensures answers are backed by your own documents, making it suitable for research papers, manuals, or any custom knowledge base.

## Install Dependencies


### This cell ensures all necessary tools are available for building the pipeline by installing or updating required Python packages:

1. **langchain:** A framework for simplifying the creation of applications that use LLMs by chaining together various components.

2. **langchain-community:** Provides integrations for external third-party resources, such as document loaders, vector stores, and specific model bindings.

3. **langchain-text-splitters:** A utility library dedicated to breaking down large documents into smaller, manageable chunks. This addresses the LLM's context length limitation.

4. **sentence-transformers:** A specialized library that provides models to convert text into fixed-size numerical vectors, known as embeddings. These embeddings are vital for the semantic search component of the Retrieval step.

5. **faiss-cpu:** A library by Facebook AI for performing efficient similarity search across large datasets of dense vectors, serving as the Vector Database in a RAG pipeline.

6. **pypdf:** A pure-Python tool used to read and extract textual data from PDF files.

In [1]:
!pip install -U langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing

## Import Modules

### This imports the necessary classes and functions from the installed libraries:

PyPDFLoader: Instantiated to load the contents of the PDF document.

RecursiveCharacterTextSplitter: The chosen implementation for breaking the input document into chunks.

SentenceTransformerEmbeddings: Used to select and load a sentence transformer model for generating semantic vectors (embeddings).

FAISS: Imported to create the in-memory index for fast similarity search across the vector embeddings.

os: A standard Python module for interfacing with the operating system, often used for file path operations.

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
import os

## Load your PDF document

Here I am using a book for builduing AI Engineering applications ( by Chip Huyen )



#### This is the first data processing step, taking the raw file and turning it into data objects:

1. pdf_path = ...: Defines the location of the source document ("AI Engineering" by Chip Huyen).

2. loader = PyPDFLoader(pdf_path): Creates a loader object tied to the document path.

3. pages = loader.load(): Executes the PDF extraction, resulting in a list where each element is a LangChain Document object corresponding to a single page from the PDF.

**Purpose:** The output confirms 991 pages were loaded, verifying successful data access and text extraction.

In [3]:
# Load PDF pages
pdf_path = "/content/_OceanofPDF.com_AI_Engineering_Building_Applications_-_Chip_Huyen.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Total pages loaded: {len(pages)}")
print("Sample page content:\n")
print(pages[3].page_content[:500])

Total pages loaded: 991
Sample page content:

AI Engineering is a comprehensive guide that serves as an essential
reference for both understanding and implementing AI systems in
practice.
—Han Lee, Director—Data Science, Moody’s
AI Engineering is an essential guide for anyone building software
with Generative AI! It demystifies the technology, highlights the
importance of evaluation, and shares what should be done to achieve
quality before starting with costly fine-tuning.
—Rafal Kawala, Senior AI Engineering Director, 16
years of experienc


#### A data inspection step to understand the overall size of the source text:

* full_text = " ".join(...): Iterates through the loaded pages and concatenates all the text into a single string.

**Purpose:** The output shows the raw document size is 1,080,309 characters. This confirms the content is too large for most LLM context windows, mandating the need for the chunking step that follows.

In [4]:
# Combine all text for exploration
full_text = " ".join([p.page_content for p in pages])
print(f"Document length: {len(full_text)} characters")
print(f"Sample snippet:\n{full_text[:600]}")

Document length: 1080309 characters
Sample snippet:
 Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping
teams bring AI into production. Drawing on her deep expertise, AI
Engineering serves as a comprehensive and holistic guide,
masterfully detailing everything required to design and deploy
generative A


## Split the document into Chunks

### This is the essential preparation for the retrieval step, optimizing the data for vector search:

1. splitter = RecursiveCharacterTextSplitter(...): Initializes the text splitting strategy.

2. chunk_size=1000: Sets the maximum size for each unit of text (chunk) at 1,000 characters.

3. chunk_overlap=200: Ensures the last 200 characters of one chunk are included in the start of the next chunk. This is a best practice to preserve context continuity and improve retrieval accuracy.

4. chunks = splitter.split_documents(pages): Executes the splitting, creating the final list of smaller, context-aware document chunks.

**Purpose:** The result, 1,640 chunks, transforms the single large document into a large set of semantically meaningful, searchable units.

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(pages)

print(f" Created {len(chunks)} chunks from {len(pages)} pages.")
print(chunks[0].page_content[:400])

 Created 1640 chunks from 991 pages.
Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping



In [6]:
print(f"Total chunks: {len(chunks)}")

Total chunks: 1640


### Access a specific chunk

#### These final cells serve only to confirm the data preparation steps succeeded:

* Accessing/Printing Chunks: Prints the first 400 characters of the first chunk, the total number of chunks, and the content of specific chunks by index (e.g., index 2, index 4, index 12).

**Purpose:** To verify that the chunks array is correctly populated and contains clean, readable, segmented text, ensuring the data is ready for the subsequent RAG steps (embedding and indexing).


In [7]:
# First chunk
print(chunks[0].page_content)

print("\n")

# Fifth chunk
print(chunks[4].page_content)

print("\n")

# 13th chunk
print(chunks[12].page_content)

Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI across the enterprise.
—Vittorio Cretella, former global CIO, P&G and Mars
Chip Huyen gets generative AI. On top of that, she is a remarkable
teacher and writer whose work has been instrumental in helping
teams bring AI into production. Drawing on her deep expertise, AI
Engineering serves as a comprehensive and holistic guide,
masterfully detailing everything required to design and deploy
generative AI applications in production.
—Luke Metz, cocreator of ChatGPT, former research
manager at OpenAI
Every AI engineer building real-world applications should read this
book. It’s a vital guide to end-to-end AI system design, from model
development and evaluation to large-scale deployment and operation.
—Andrei Lopatenko, Director Search and AI, Neuron7


AI Engineering
Building Applications with Foun

### Loop Through all chunks

If you want to quickly see the first 200 characters of each chunk:

In [8]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content[:200])
    print("\n")

--- Chunk 1 ---
Praise for AI Engineering
This book offers a comprehensive, well-structured guide to the
essential aspects of building generative AI systems. A must-read for
any professional looking to scale AI acros


--- Chunk 2 ---
This book serves as an essential guide for building AI products that
can scale. Unlike other books that focus on tools or current trends
that are constantly changing, Chip delivers timeless foundation


--- Chunk 3 ---
AI Engineering is a practical guide that provides the most up-to-date
information on AI development, making it approachable for novice
and expert leaders alike. This book is an essential resource for



--- Chunk 4 ---
AI Engineering is a comprehensive guide that serves as an essential
reference for both understanding and implementing AI systems in
practice.
—Han Lee, Director—Data Science, Moody’s
AI Engineering is


--- Chunk 5 ---
AI Engineering
Building Applications with Foundation Models
Chip Huyen
OceanofPDF .com


--- Chunk 6 ---
AI 

### Access a chunk for later use

In [9]:
example_chunk = chunks[2].page_content
print(example_chunk)

AI Engineering is a practical guide that provides the most up-to-date
information on AI development, making it approachable for novice
and expert leaders alike. This book is an essential resource for
anyone looking to build robust and scalable AI systems.
—Vicki Reyzelman, Chief AI Solutions Architect,
Mave Sparks


## Create Embeddings

We’ll use a pre-trained sentence embedding model from Hugging Face — "sentence-transformers/all-MiniLM-L6-v2" (lightweight and fast).

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Test embedding creation on one sample chunk
test_embedding = embedding_model.embed_query(chunks[0].page_content)
print(f" Sample embedding vector length: {len(test_embedding)}")

/tmp/ipython-input-599179408.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Sample embedding vector length: 384


#### We’ll encode each chunk into a vector representation.

In [24]:
texts = [c.page_content for c in chunks]
embeddings = embedding_model.embed_documents(texts)

print(" Generated embeddings with shape:", len(embeddings), "x", len(embeddings[0]))

 Generated embeddings with shape: 1640 x 384


#### embed_documents() returns a list of embeddings rather than a NumPy array, so before adding to FAISS you’ll need:



In [27]:
import numpy as np
embeddings = np.array(embeddings, dtype = "float32")

## Build a FAISS Vector Store

FAISS (Facebook AI Similarity Search) helps us store and quickly find similar vectors (chunks).

In [28]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product = cosine similarity for normalized vectors
index.add(embeddings.astype("float32"))

print(f" FAISS index created with {index.ntotal} vectors.")

 FAISS index created with 1640 vectors.


## Save the FAISS Index & Metadata

We’ll store:

	•	The FAISS binary index file
	•	The chunk metadata (page numbers, chunk IDs, etc.)

In [29]:
import json

os.makedirs("faiss_index_ai_engineering", exist_ok=True)
faiss.write_index(index, "faiss_index_ai_engineering/index.faiss")

metadata = [c.metadata for c in chunks]
with open("faiss_index_ai_engineering/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(" Saved FAISS index and metadata locally.")

 Saved FAISS index and metadata locally.


## Test Semantic Search

Let’s test if your RAG base is working properly.

In [33]:
def semantic_search(query, k=3):
    # Use LangChain method for embeddings
    query_emb = embedding_model.embed_query(query)
    query_emb = np.array([query_emb], dtype="float32")

    # Perform FAISS search
    distances, indices = index.search(query_emb, k)

    # Collect results
    results = []
    for i, idx in enumerate(indices[0]):
        text = chunks[idx].page_content[:400].replace("\n", " ")
        page = chunks[idx].metadata.get("page", "Unknown")
        results.append({"rank": i+1, "page": page, "text": text})

    return results

### Define your queries

These are natural language questions you want to ask your document (in this case, Chip Huyen’s AI Engineering book).

In [34]:
# Example user queries
queries = [
    "What is AI engineering?",
    "How does the book describe model deployment?",
    "What are some challenges in real-world machine learning systems?",
]

###Run the search for each query

We’ll loop through each question and show the top k (default 3) most relevant chunks.

In [35]:
for q in queries:
    print(f"\n Query: {q}\n" + "-"*80)
    results = semantic_search(q, k=3)

    for r in results:
        print(f" Rank {r['rank']} | Page {r['page']}")
        print(r['text'])
        print("-"*80)


 Query: What is AI engineering?
--------------------------------------------------------------------------------
 Rank 1 | Page 5
AI Engineering by Chip Huyen Copyright © 2025 Developer Experience Advisory LLC. All rights reserved. Printed in the United States of America. Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472. O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are also available for most titles (http://oreilly.com). For more i
--------------------------------------------------------------------------------
 Rank 2 | Page 2
AI Engineering is a practical guide that provides the most up-to-date information on AI development, making it approachable for novice and expert leaders alike. This book is an essential resource for anyone looking to build robust and scalable AI systems. —Vicki Reyzelman, Chief AI Solutions Architect, Mave Sparks
------------------------------------------------------